# Train TF-IDF, CodeBERT và ASTNN trên Google Colab

Notebook này chạy các script model hiện có trong thư mục `scripts/` và lưu checkpoint vào `artifacts/code-models/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

# Đổi tên thư mục nếu bạn đặt gói ở vị trí khác trong MyDrive.
PROJECT_ROOT = Path('/content/drive/MyDrive/Colab_Training')
os.chdir(PROJECT_ROOT)
print('Project:', PROJECT_ROOT)

In [ ]:
!pip install -q -r requirements-ml.txt
!nvidia-smi

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Chưa bật GPU. Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại.')
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

DATASET = PROJECT_ROOT / 'Dataset' / 'Project_CodeNet'
required_dirs = [
    DATASET / 'problem_descriptions',
    DATASET / 'derived' / 'input_output',
    DATASET / 'metadata',
    DATASET / 'data',
]
missing = [path for path in required_dirs if not path.is_dir()]
if missing:
    raise FileNotFoundError(f'Thiếu thư mục CodeNet: {missing}')
print('Dataset folder:', DATASET)
print('Problem descriptions:', len(list((DATASET / 'problem_descriptions').glob('p*.html'))))
print('Metadata files:', len(list((DATASET / 'metadata').glob('p*.csv'))))
print('Source folders:', len(list((DATASET / 'data').glob('p*'))))

In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    'scripts/train_all_models.py',
    '--dataset', str(DATASET),
    '--output-dir', str(PROJECT_ROOT / 'artifacts' / 'code-models'),
    '--epochs', '3',
    '--batch-size', '2',
    '--max-length', '512',
    '--device', 'cuda',
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json

report_path = PROJECT_ROOT / 'artifacts' / 'code-models' / 'comparison.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
for result in report['results']:
    print(result['model'])
    print('  accuracy:', result.get('accuracy'))
    print('  macro_f1:', result.get('macro_f1'))